In [1]:
# Settings - change these to match your project
PROJECT_ID = "qwiklabs-gcp-01-5fc8b912fc6a"   # your GCP project
DATASET = "fraud_detection"           # created below if it does not exist
LOCATION = "US"
GCS_URI = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

from google.cloud import bigquery

# connect to bigquery
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# create the dataset if it isn't there yet
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")

dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print("Dataset ready:", DATASET)

Dataset ready: fraud_detection


In [2]:
# load the raw csv into bigquery
raw_table = f"{PROJECT_ID}.{DATASET}.fraud_data_raw"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)

load_job = client.load_table_from_uri(GCS_URI, raw_table, job_config=job_config)
load_job.result()

print("Loaded rows:", client.get_table(raw_table).num_rows)

Loaded rows: 50000


In [7]:
import pandas as pd
import numpy as np

df = client.query(f"SELECT * FROM `{raw_table}`").to_dataframe()
df.head()

,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0


In [8]:
# one-hot the two category columns
df = pd.get_dummies(df, columns=["Employment_Status", "Device_Type"])

# age buckets, then one-hot them
df["age_group"] = pd.cut(df["Age"], [18, 25, 35, 45, 55, 65, np.inf], right=False,
                         labels=["18_24", "25_34", "35_44", "45_54", "55_64", "65_plus"])
df = pd.get_dummies(df, columns=["age_group"])

# income / amount ratio
df["Income_to_Amount_Requested"] = df["Income"] / df["Amount_Requested"]

# days between the previous assistance and this application
df["Application_Date"] = pd.to_datetime(df["Application_Date"])
df["Previous_Assistance_Date"] = pd.to_datetime(df["Previous_Assistance_Date"])
df["Time_Since_Previous_Assistance"] = (df["Application_Date"] - df["Previous_Assistance_Date"]).dt.days

# True/False columns -> 1/0
bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,...,Device_Type_Mobile,Device_Type_Tablet,age_group_18_24,age_group_25_34,age_group_35_44,age_group_45_54,age_group_55_64,age_group_65_plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance
0,217,65,28984,4,5872,0,NaT,0,1,156.133.45.45,...,1,0,0,0,0,0,0,1,4.935967,NaN
1,226,54,0,1,6631,0,NaT,0,1,245.13.80.245,...,0,1,0,0,0,1,0,0,0.0,NaN
2,240,26,64477,5,8612,0,NaT,1,1,213.103.170.95,...,1,0,0,1,0,0,0,0,7.486879,NaN
3,252,28,28576,4,2951,0,NaT,1,1,234.179.149.207,...,0,0,0,1,0,0,0,0,9.683497,NaN
4,266,43,44930,5,2324,0,NaT,0,1,66.109.96.227,...,1,0,0,0,1,0,0,0,19.333046,NaN


In [9]:
train_table = f"{PROJECT_ID}.{DATASET}.fraud_training_data"

client.load_table_from_dataframe(
    df,
    train_table,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
).result()

print("saved ->", train_table)

saved -> qwiklabs-gcp-01-5fc8b912fc6a.fraud_detection.fraud_training_data


In [10]:
client.query(f"SELECT * FROM `{train_table}` LIMIT 10").to_dataframe()


,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,...,Device_Type_Mobile,Device_Type_Tablet,age_group_18_24,age_group_25_34,age_group_35_44,age_group_45_54,age_group_55_64,age_group_65_plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance
0,1750,18,0,4,7198,0,NaT,1,1,134.91.210.32,...,1,0,1,0,0,0,0,0,0.000000,NaN
1,2072,18,0,3,6160,0,NaT,0,1,66.109.187.95,...,0,1,1,0,0,0,0,0,0.000000,NaN
2,3104,18,45659,4,1650,0,NaT,0,1,172.12.109.68,...,0,1,1,0,0,0,0,0,27.672121,NaN
3,4500,18,39485,3,6242,0,NaT,1,1,172.12.132.176,...,1,0,1,0,0,0,0,0,6.325697,NaN
4,4847,18,0,2,8217,0,NaT,0,1,176.235.245.243,...,0,1,1,0,0,0,0,0,0.000000,NaN
5,5198,18,17131,5,8648,0,NaT,1,1,19.29.10.179,...,0,1,1,0,0,0,0,0,1.980920,NaN
6,6175,18,0,3,5029,0,NaT,1,1,184.225.204.231,...,0,1,1,0,0,0,0,0,0.000000,NaN
7,8384,18,49981,4,9572,0,NaT,1,1,245.13.15.132,...,1,0,1,0,0,0,0,0,5.221584,NaN
8,9165,18,39596,1,3676,0,NaT,0,1,22.187.15.226,...,0,1,1,0,0,0,0,0,10.771491,NaN
9,11938,18,0,5,2844,0,NaT,0,1,184.225.53.228,...,0,0,1,0,0,0,0,0,0.000000,NaN


In [11]:
# sanity check: rows where the applicant had previous assistance
client.query(f"""
  SELECT Applicant_ID, Previous_Assistance_Date, Application_Date, Time_Since_Previous_Assistance
  FROM `{train_table}`
  WHERE Time_Since_Previous_Assistance IS NOT NULL
  LIMIT 10
""").to_dataframe()

,Applicant_ID,Previous_Assistance_Date,Application_Date,Time_Since_Previous_Assistance
0,28640,2022-10-05,2024-01-18,470.0
1,37562,2022-11-20,2024-04-03,500.0
2,6875,2023-01-05,2024-03-01,421.0
3,539,2023-01-19,2024-03-22,428.0
4,2973,2023-01-19,2024-05-31,498.0
5,10623,2023-01-30,2024-05-09,465.0
6,9059,2023-02-25,2024-05-09,439.0
7,13098,2023-03-10,2024-05-19,436.0
8,1523,2023-03-19,2024-06-22,461.0
9,13038,2023-05-08,2024-08-05,455.0
